In [1]:
import os, time, random, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm  # ✅ auto chooses notebook-safe or terminal-safe mode
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from albumentations import Compose, Resize, Normalize, HorizontalFlip, RandomBrightnessContrast
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from efficientnet_pytorch import EfficientNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ Device:", device)
torch.backends.cudnn.benchmark = True


✅ Device: cuda


In [2]:
class ChestXrayDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df=df.reset_index(drop=True)
        self.img_dir=img_dir
        self.transform=transform
        self.labels=['Atelectasis','Consolidation','Infiltration','Pneumothorax','Edema',
                     'Emphysema','Fibrosis','Effusion','Pneumonia','Pleural_Thickening',
                     'Cardiomegaly','Nodule','Mass','Hernia']
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        img_path=None
        for j in range(1,13):
            p=os.path.join(self.img_dir,f'images_{j:03d}/images',row['Image Index'])
            if os.path.exists(p): img_path=p; break
        img=np.array(Image.open(img_path).convert("RGB"))
        y=np.zeros(len(self.labels),dtype=np.float32)
        if row['Finding Labels']!="No Finding":
            for k,l in enumerate(self.labels):
                if l in row['Finding Labels']: y[k]=1
        if self.transform:
            img=self.transform(image=img)['image']
        return img,torch.tensor(y)


In [3]:
extract="./chest_xray_data/"
data=pd.read_csv(os.path.join(extract,"Data_Entry_2017.csv"))

train_df,val_df,test_df=np.split(
    data.sample(frac=1,random_state=42),
    [int(.7*len(data)),int(.85*len(data))])

tf_train=Compose([
    Resize(224,224),
    HorizontalFlip(p=0.5),
    RandomBrightnessContrast(0.15,0.15,p=0.3),
    Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
    ToTensorV2()
])
tf_val=Compose([
    Resize(224,224),
    Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
    ToTensorV2()
])

train_loader=DataLoader(ChestXrayDataset(train_df,extract,tf_train),
                        batch_size=16,shuffle=True,num_workers=4,pin_memory=True)
val_loader=DataLoader(ChestXrayDataset(val_df,extract,tf_val),
                      batch_size=16,shuffle=False,num_workers=4,pin_memory=True)
print(f"✅ Train: {len(train_loader)} | Val: {len(val_loader)}")


✅ Train: 4906 | Val: 1052


C:\Users\91850\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [4]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1,gamma=2):
        super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self, inputs, targets):
        BCE=F.binary_cross_entropy_with_logits(inputs,targets,reduction='none')
        pt=torch.exp(-BCE)
        return (self.alpha*(1-pt)**self.gamma*BCE).mean()


In [5]:
class EffNetB3(nn.Module):
    def __init__(self,num_classes=14):
        super().__init__()
        self.net=EfficientNet.from_pretrained('efficientnet-b3')
        n=self.net._fc.in_features
        self.net._fc=nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(n,num_classes)
        )
    def forward(self,x): return self.net(x)

model=EffNetB3().to(device)
print("✅ Model loaded (EfficientNet‑B3)")


Loaded pretrained weights for efficientnet-b3
✅ Model loaded (EfficientNet‑B3)


In [6]:
scaler=GradScaler(device='cuda')

def train_epoch(model,ldr,loss_fn,opt,scaler):
    model.train(); tot_loss=0;labs,preds=[],[]
    for x,y in tqdm(ldr,desc="Training",total=len(ldr),ncols=100):
        x,y=x.to(device),y.to(device)
        opt.zero_grad(set_to_none=True)
        with autocast(device_type='cuda',dtype=torch.float16,enabled=True):
            o=model(x)
            loss=loss_fn(o,y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        tot_loss+=loss.item()
        labs.append(y.detach().cpu().numpy())
        preds.append(torch.sigmoid(o.detach()).cpu().numpy())
    auc=roc_auc_score(np.vstack(labs),np.vstack(preds),average='macro')
    return tot_loss/len(ldr),auc

def val_epoch(model,ldr,loss_fn):
    model.eval(); tot_loss=0;labs,preds=[],[]
    with torch.no_grad():
        for x,y in tqdm(ldr,desc="Validating",total=len(ldr),ncols=100):
            x,y=x.to(device),y.to(device)
            with autocast(device_type='cuda',dtype=torch.float16,enabled=True):
                o=model(x); loss=loss_fn(o,y)
            tot_loss+=loss.item()
            labs.append(y.cpu().numpy())
            preds.append(torch.sigmoid(o).cpu().numpy())
    auc=roc_auc_score(np.vstack(labs),np.vstack(preds),average='macro')
    return tot_loss/len(ldr),auc


In [7]:
from tqdm.auto import tqdm
import time
for i in tqdm(range(15), desc="Check Bar"):
    time.sleep(0.1)


Check Bar:   0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-5)
loss_fn=FocalLoss()
best_auc=0
for epoch in range(10):
    print(f"\n{'='*70}\nEPOCH [{epoch+1}/10]\n{'='*70}")
    trL,trA=train_epoch(model,train_loader,loss_fn,opt,scaler)
    vlL,vlA=val_epoch(model,val_loader,loss_fn)
    print(f"📊 Train Loss={trL:.4f} AUC={trA:.4f} | Val Loss={vlL:.4f} AUC={vlA:.4f}")
    if vlA>best_auc:
        best_auc=vlA; torch.save(model.state_dict(),"best_effnetB3.pth")
        print(f"✅ Best Model Saved (AUC={best_auc:.4f})")


EPOCH [1/10]
